# 📘 Notebook 5 — Alberi Decisionali e Random Forest

Finora abbiamo visto modelli **lineari**. Ora vediamo modelli che imparano **regole if-then-else**, molto intuitivi e potenti: gli **alberi decisionali**.

## 🎯 Cosa imparerai qui
1. Come funziona un **albero decisionale** (intuizione visuale)
2. Implementazione con scikit-learn per **classificazione e regressione**
3. Il problema dell'**overfitting** e come limitarlo
4. Le **Random Forest**: la potenza dell'ensemble
5. **Feature importance**: capire cosa il modello considera importante
6. Quando usare alberi vs modelli lineari

## 🤔 Quando usare gli alberi?
- Quando i dati hanno **relazioni non lineari** complesse
- Quando NON vuoi pre-processare troppo (alberi non richiedono scaling)
- Quando vuoi un modello **interpretabile** (puoi visualizzare l'albero!)
- Su dati **misti** (numerici e categorici)

Esempi: predire la sopravvivenza al Titanic, valutare il rischio di un prestito, classificare specie di fiori.


![albero](Media/05/albero.png)


## 1️⃣ L'idea: imparare a fare domande

Un albero decisionale è come un **gioco delle 20 domande**: pone domande sulle caratteristiche per arrivare alla risposta.

**Esempio: "Devo prendere l'ombrello?"**
```
                    [ Piove ora? ]
                       /     \
                     SÌ       NO
                     /         \
             [ombrello]    [Nuvolo?]
                            /    \
                           SÌ    NO
                          /       \
                  [forse omb.]   [niente]
```

Il modello impara **automaticamente** quali domande fare e in che ordine, basandosi sui dati.


![anatomia](Media/05/anatomia.png)


In [ ]:
# Carichiamo il famoso dataset Iris (3 specie di fiori)
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

print(f"Forma: {X.shape}")
print(f"Specie: {iris.target_names}")
print("\nPrime righe:")
print(X.head())
print("\nDistribuzione classi:", y.value_counts().to_dict())

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Creiamo un albero SEMPLICE (max_depth=3)
albero = DecisionTreeClassifier(max_depth=3, random_state=42)
albero.fit(X_train, y_train)

# Visualizziamo l'albero!
plt.figure(figsize=(16, 8))
plot_tree(
    albero,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,           # colora i nodi per classe
    rounded=True,
    fontsize=10
)
plt.title('Albero decisionale per il dataset Iris')
plt.show()

# 💡 Come leggerlo:
# - Ogni nodo fa una DOMANDA (es: 'petal length <= 2.45')
# - Se SÌ vai a sinistra, se NO a destra
# - Le foglie (in basso) contengono la previsione
# - 'gini' misura quanto è 'pura' la divisione (0 = perfetta)

In [ ]:
# Performance del modello
from sklearn.metrics import accuracy_score, classification_report

y_pred = albero.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\n", classification_report(y_test, y_pred, target_names=iris.target_names))

## 2️⃣ Come l'albero "divide" lo spazio

Visualizziamo come l'albero crea **regioni rettangolari** nello spazio delle caratteristiche.

![divisione](Media/05/divisione.png)


In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

# Per visualizzare in 2D, usiamo solo 2 caratteristiche (petal length e width)
X2 = X[['petal length (cm)', 'petal width (cm)']].values
albero2 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X2, y)

# Creiamo una griglia fitta di punti
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

# Predizione su tutta la griglia
Z = albero2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF9999', '#99FF99', '#9999FF']))
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap=ListedColormap(['red', 'green', 'blue']), edgecolor='k')
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.title("Le 'regioni di decisione' di un albero (sono RETTANGOLI!)")
plt.show()

# 💡 Le frontiere decisionali degli alberi sono SEMPRE perpendicolari agli assi.
# Differenza con la regressione logistica, che fa frontiere oblique (lineari).

## 3️⃣ Il pericolo dell'OVERFITTING

Se lasciamo l'albero crescere senza limiti, **memorizza** i dati di training invece di generalizzare. Questo è l'**overfitting**.

Sintomo: **alta accuracy sul training, bassa sul test**.


![profondita](Media/05/profondita.png)


In [ ]:
# Confrontiamo alberi con profondità diverse
profondita = [1, 2, 3, 5, 10, None]   # None = nessun limite

print(f"{'Depth':<8}{'Train acc':<12}{'Test acc':<12}{'Gap (overfitting)':<20}")
print("-" * 50)

for d in profondita:
    a = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc = a.score(X_train, y_train)
    test_acc  = a.score(X_test, y_test)
    gap = train_acc - test_acc
    label = str(d) if d is not None else 'None (∞)'
    print(f"{label:<8}{train_acc:<12.3f}{test_acc:<12.3f}{gap:<20.3f}")

# 💡 Con profondità ∞:
# - Training accuracy = 1.0 (perfetto, ha 'memorizzato')
# - Test accuracy può scendere → OVERFITTING!
# Per evitarlo, limitiamo la complessità.

### Tecniche anti-overfitting per gli alberi ("pruning")

- `max_depth`: profondità massima
- `min_samples_split`: minimo di esempi per dividere un nodo
- `min_samples_leaf`: minimo di esempi in una foglia
- `max_features`: quante caratteristiche valutare per ogni split

In [ ]:
# Esempio: confronto tra albero non potato (overfit) e albero potato
albero_overfit = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
albero_potato  = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42).fit(X_train, y_train)

print(f"Non potato:  train={albero_overfit.score(X_train, y_train):.3f}, test={albero_overfit.score(X_test, y_test):.3f}")
print(f"Potato:      train={albero_potato.score(X_train, y_train):.3f},  test={albero_potato.score(X_test, y_test):.3f}")

# 💡 L'albero potato spesso ha test accuracy MIGLIORE perché generalizza di più.

## 4️⃣ Random Forest: la forza del gruppo

Un singolo albero è **fragile**: piccole variazioni nei dati → albero molto diverso.

**Idea della Random Forest**: 
- Costruisco **TANTI alberi** (es: 100), ognuno su un sotto-campione casuale dei dati e con un sotto-campione delle caratteristiche
- Per fare una previsione, **chiedo a tutti** e prendo:
  - la **classe più votata** (per classificazione)
  - la **media** (per regressione)

🎯 Risultato: predizioni più stabili e accurate. È uno degli algoritmi più usati nella pratica!


![randomforest](Media/05/randomforest.png)

![bagging](Media/05/bagging.png)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest con 100 alberi
rf = RandomForestClassifier(
    n_estimators=100,    # numero di alberi
    max_depth=5,         # profondità massima per ogni albero
    random_state=42,
    n_jobs=-1            # usa tutti i CPU disponibili (più veloce)
)
rf.fit(X_train, y_train)

print(f"Random Forest: train={rf.score(X_train, y_train):.3f}, test={rf.score(X_test, y_test):.3f}")
print(f"Singolo albero potato: train={albero_potato.score(X_train, y_train):.3f}, test={albero_potato.score(X_test, y_test):.3f}")

### 4.1 Esempio più sfidante: dataset "Wine"

In [ ]:
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.3, random_state=42, stratify=yw)

# Confronto fra 3 modelli
modelli = {
    'Regressione Logistica': LogisticRegression(max_iter=5000),
    'Albero Decisionale':    DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# La regressione logistica vuole dati scalati
scaler = StandardScaler()
Xw_train_s = scaler.fit_transform(Xw_train)
Xw_test_s  = scaler.transform(Xw_test)

print(f"{'Modello':<25}{'Train':<10}{'Test':<10}")
print("-" * 45)
for nome, m in modelli.items():
    if 'Logistica' in nome:
        m.fit(Xw_train_s, yw_train)
        tr, te = m.score(Xw_train_s, yw_train), m.score(Xw_test_s, yw_test)
    else:
        m.fit(Xw_train, yw_train)   # gli alberi non hanno bisogno di scaling!
        tr, te = m.score(Xw_train, yw_train), m.score(Xw_test, yw_test)
    print(f"{nome:<25}{tr:<10.3f}{te:<10.3f}")

## 5️⃣ Feature Importance: cosa il modello ritiene importante?

Una grande forza degli alberi (e delle Random Forest) è che ci dicono **quali caratteristiche contano di più**.

![variabili](Media/05/variabili.png)

In [ ]:
rf_wine = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(Xw_train, yw_train)

# Estraiamo le importanze
importanze = pd.Series(rf_wine.feature_importances_, index=wine.feature_names).sort_values()

plt.figure(figsize=(10, 6))
importanze.plot(kind='barh', color='teal')
plt.title('Importanza delle caratteristiche (Random Forest sul dataset Wine)')
plt.xlabel('Importanza')
plt.grid(True, alpha=0.3, axis='x')
plt.show()

# 💡 Possiamo capire QUALI variabili guidano le previsioni.
# Utile per:
# - capire il dominio
# - eliminare caratteristiche inutili
# - spiegare il modello a chi non è tecnico

## 6️⃣ Random Forest per la REGRESSIONE

Funziona benissimo anche per prevedere numeri, non solo classi.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Generiamo dati con pattern NON LINEARE (quello che la regressione lineare non sa gestire bene)
np.random.seed(42)
x = np.linspace(-3, 3, 200)
y_vero = np.sin(x * 2) * 3 + x**2 * 0.5 + np.random.normal(0, 0.5, len(x))

X = x.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y_vero, test_size=0.3, random_state=42)

from sklearn.linear_model import LinearRegression

# Confrontiamo regressione lineare e Random Forest
lin = LinearRegression().fit(X_train, y_train)
rf  = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

x_grid = np.linspace(-3, 3, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_train, y_train, alpha=0.4, s=20, label='Training')
axes[0].plot(x_grid, lin.predict(x_grid), 'r-', linewidth=2, label='Regressione lineare')
axes[0].set_title(f'Regressione lineare\nR² test = {lin.score(X_test, y_test):.3f}')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(X_train, y_train, alpha=0.4, s=20, label='Training')
axes[1].plot(x_grid, rf.predict(x_grid), 'g-', linewidth=2, label='Random Forest')
axes[1].set_title(f'Random Forest\nR² test = {rf.score(X_test, y_test):.3f}')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 La RF cattura il pattern non lineare!
# Nota però: la RF fa previsioni 'a gradini' (combinazione di alberi).

## 7️⃣ Esempio completo: predire la sopravvivenza al Titanic

Un esempio classico con dati misti.

In [ ]:
# Creiamo un dataset Titanic semplificato
np.random.seed(42)
n = 500
df_t = pd.DataFrame({
    'eta':       np.random.normal(30, 14, n).clip(1, 80),
    'sesso':     np.random.choice(['M', 'F'], n, p=[0.65, 0.35]),
    'classe':    np.random.choice([1, 2, 3], n, p=[0.2, 0.2, 0.6]),
    'tariffa':   np.random.exponential(30, n).clip(5, 500),
    'familiari': np.random.poisson(0.5, n),
})

# La sopravvivenza dipende (in modo realistico) da:
# - donne e bambini più probabili sopravvissuti
# - prima classe più probabile
prob_surv = (
    0.05
    + 0.45 * (df_t['sesso'] == 'F').astype(int)
    + 0.20 * (df_t['classe'] == 1).astype(int)
    + 0.10 * (df_t['classe'] == 2).astype(int)
    + 0.15 * (df_t['eta'] < 12).astype(int)
).clip(0.05, 0.95)
df_t['sopravvissuto'] = (np.random.random(n) < prob_surv).astype(int)

df_t.head()

In [ ]:
# Pipeline: encoding + split + RF
X = pd.get_dummies(df_t.drop('sopravvissuto', axis=1), columns=['sesso'], dtype=int)
y = df_t['sopravvissuto']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

from sklearn.metrics import classification_report, confusion_matrix
y_pred = rf.predict(X_test)

print("📊 Performance:")
print(classification_report(y_test, y_pred, target_names=['Non sopravvissuto', 'Sopravvissuto']))

# Feature importance
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 4), color='steelblue')
plt.title('Cosa conta per sopravvivere?')
plt.show()

## 🎓 Riepilogo del Notebook 5

![vs](Media/05/vs.png)

### Albero Decisionale
- ✅ Intuitivo (visualizzabile), non richiede scaling, gestisce dati misti
- ❌ Tendenza all'overfitting, instabile (piccole variazioni → albero molto diverso)

### Random Forest
- ✅ Molto accurato, robusto, fornisce feature importance
- ✅ Funziona quasi sempre "out of the box" anche senza tuning
- ❌ Più lento del singolo albero, meno interpretabile

### 🤔 Quando preferire alberi/foreste a modelli lineari
- Pattern complessi non lineari
- Dati tabellari misti (numerici + categorici)
- Vuoi capire l'importanza delle variabili
- Non vuoi pre-processare troppo

### 🤔 Quando preferire modelli lineari
- Pattern (quasi) lineari
- Ti serve la **probabilità** ben calibrata
- Pochi dati di training (gli alberi vanno in overfit più facilmente)
- Vuoi un modello "matematico" interpretabile classico

👉 **Prossimo notebook (06)**: il **Clustering** e il **K-Means** — il primo algoritmo di **apprendimento NON supervisionato** (senza target!).